## 时间特征可视化

In [ ]:
import pandas as pd

#### 读取数据

In [ ]:
# 读取文件，相对路径可能需调整为实际路径
input_file = "..\\data\\CICAPT_IIOT\\Provenance_Logs\\Phase1_Provenance.csv"

# 读取全部数据
# 自动检测分隔符，如果出错可以加delimiter=','或delimiter='\t'
df = pd.read_csv(input_file, low_memory=False)


# 合并时间戳字段，优先级为: time > seen time > start time
def extract_time(row):
    for field in ["seen time", "start time", "time"]:
        if pd.notnull(row.get(field)):
            return row.get(field)
    return None


# 新建time列
df["time"] = df.apply(extract_time, axis=1)

# 只保留所需字段
fields = ["id", "time", "label"]
exists_fields = [f for f in fields if f in df.columns]
df = df[exists_fields]

# 统计条目数
num_rows = len(df)

print(f"已读取条目数: {num_rows}")
print(df.head())

In [ ]:
# 将time列转为数字类型，缺失为NaN
df['time'] = pd.to_numeric(df['time'], errors='coerce')

# 先丢弃time字段为NaN的所有数据
orig_len = len(df)
df = df.dropna(subset=['time'])
print(f"去除time缺失前条数: {orig_len}，去除后条数: {len(df)}")

# 归一化：所有时间减去最小值，四舍五入为整数
time_min = df['time'].min()
df['time_norm'] = (df['time'] - time_min).round().astype(int)

print(f"最终有效条目数: {len(df)}")
print(df.head())

#### 绘图

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes, mark_inset

In [ ]:
# label归一化为整数（有些数据可能是字符串）
df['label'] = pd.to_numeric(df['label'], errors='coerce').fillna(0).astype(int)

# 设置颜色映射：0=淡灰色，1=红色
colors = np.where(df['label'] == 1, '#e41a1c', '#cccccc')

plt.figure(figsize=(10, 3))
plt.scatter(df['time_norm'], [1]*len(df), c=colors, marker='|', s=100)
plt.xlabel('Normalized Time')
plt.yticks([])
plt.title('Event Distribution over Time (Gray=0, Red=1)')

# 只画简易图例
import matplotlib.lines as mlines
plt.legend(handles=[
    mlines.Line2D([], [], color='#cccccc', marker='|', linestyle='None', markersize=15, label='Label=0'),
    mlines.Line2D([], [], color='#e41a1c', marker='|', linestyle='None', markersize=15, label='Label=1'),
], loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# ===== 数据准备（可选，若已完成可忽略） =====
df['label'] = pd.to_numeric(df['label'], errors='coerce').fillna(0).astype(int)
# 按时间排序，显示更规整
_df = df.sort_values('time_norm').reset_index(drop=True)

# 统一的颜色
COLOR_NORMAL = '#cccccc'  # 淡灰
COLOR_MAL = '#e41a1c'     # 红

# =============================
# 图1：事件时间轴（Timeline / Eventplot）
# =============================
plt.figure(figsize=(12, 2.2))
# eventplot 接受一维数组，按类别分别绘制，重叠在同一条线上
plt.eventplot(_df.loc[_df['label']==0, 'time_norm'].to_numpy(),
              lineoffsets=0.5, linelengths=0.6, colors=COLOR_NORMAL, linewidths=1)
plt.eventplot(_df.loc[_df['label']==1, 'time_norm'].to_numpy(),
              lineoffsets=0.5, linelengths=0.6, colors=COLOR_MAL, linewidths=1)

plt.xlabel('Normalized Time')
plt.yticks([])
plt.title('Event Timeline (Gray=0, Red=1)')

# 简洁图例
import matplotlib.lines as mlines
plt.legend(handles=[
    mlines.Line2D([], [], color=COLOR_NORMAL, marker='|', linestyle='None', markersize=12, label='Label=0'),
    mlines.Line2D([], [], color=COLOR_MAL, marker='|', linestyle='None', markersize=12, label='Label=1'),
], loc='upper right', frameon=False)

plt.tight_layout()
plt.show()

# =============================
# 图2：KDE 密度曲线（按类别对比）
# =============================
plt.figure(figsize=(12, 3.2))
# 只对横轴（time_norm）做密度估计；注意 KDE 需要较多点才稳定
sns.kdeplot(data=_df.loc[_df['label']==0, 'time_norm'], fill=True, alpha=0.25, color=COLOR_NORMAL, label='Label=0')
sns.kdeplot(data=_df.loc[_df['label']==1, 'time_norm'], fill=True, alpha=0.25, color=COLOR_MAL, label='Label=1')

# 可选：在密度曲线下方加 rug 显示原始点（点多时可注释）
sns.rugplot(x=_df.loc[_df['label']==0, 'time_norm'], height=0.03, color=COLOR_NORMAL, alpha=0.5)
sns.rugplot(x=_df.loc[_df['label']==1, 'time_norm'], height=0.03, color=COLOR_MAL, alpha=0.7)

plt.xlabel('Normalized Time')
plt.ylabel('Density')
plt.title('KDE Density over Time (Gray=0, Red=1)')
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

# ===== 基本输入 =====
TARGET_SNAPSHOTS = 35    # 目标快照数
BURST_FRAC = 0         # 爆发区：前 2/3 时间

# 1) 排序 & 定义两段
df_sorted = df.sort_values('time_norm').reset_index(drop=True)
t = df_sorted['time_norm'].to_numpy()
Tmin, Tmax = t[0], t[-1]
Tburst = Tmin + BURST_FRAC * (Tmax - Tmin)

# 按“时间阈值”分两段索引
idx_burst_end = np.searchsorted(t, Tburst, side='left')  # [0, idx_burst_end) 属于爆发区
n_total = len(t)
n_burst = idx_burst_end
n_calm  = n_total - n_burst

# 2) 按事件占比分配快照数
snap_burst = max(1, int(round(TARGET_SNAPSHOTS * n_burst / n_total)))
snap_calm  = max(1, TARGET_SNAPSHOTS - snap_burst)

print(f"[Info] total={n_total}, burst_evts={n_burst}, calm_evts={n_calm}")
print(f"[Info] target snapshots: burst={snap_burst}, calm={snap_calm} (sum={snap_burst+snap_calm})")

# 3) 在各自分区做“等事件数”切分（按索引均分）
def equal_count_edges(n_events, k_windows, start_idx=0):
    """
    把[start_idx, start_idx+n_events) 这段索引均分为k段（尽量均匀），
    返回每段的 (idx_start, idx_end) 半开区间。
    """
    # 基础配额与余数分配：前 r 段多 1 条
    base = n_events // k_windows
    rem  = n_events %  k_windows
    sizes = [base + (1 if i < rem else 0) for i in range(k_windows)]
    edges = []
    cur = start_idx
    for sz in sizes:
        edges.append((cur, cur + sz))
        cur += sz
    return edges  # k 个 (s,e)

# 爆发区等事件数
edges_burst_idx = equal_count_edges(n_burst, snap_burst, start_idx=0)
# 平稳区等事件数（从 idx_burst_end 开始继续切）
edges_calm_idx  = equal_count_edges(n_calm,  snap_calm,  start_idx=idx_burst_end)

# 4) 映射到时间边界
def idx_to_time_windows(edges_idx, time_array):
    wins = []
    for s, e in edges_idx:
        if e <= s: 
            continue
        # 用半开区间 [t[s], t[e-1]] 表示；最后一个窗右端补到 Tmax
        t_start = float(time_array[s])
        t_end   = float(time_array[e-1])  # 右端对齐到该段最后一个事件的时间
        wins.append((t_start, t_end, e - s))
    return wins

wins_burst = idx_to_time_windows(edges_burst_idx, t)
wins_calm  = idx_to_time_windows(edges_calm_idx,  t)

# 5) 汇总结果（保持时间顺序）
final_windows = wins_burst + wins_calm
out = pd.DataFrame(final_windows, columns=['t_start', 't_end', 'n_events'])

# 兜底：最后一个窗的 t_end 对齐到全局 Tmax（可要可不要）
out.loc[len(out)-1, 't_end'] = max(out.iloc[-1]['t_end'], Tmax)
out['duration'] = out['t_end'] - out['t_start']

print(f"[Result] snapshots={len(out)} (target={TARGET_SNAPSHOTS})")
print(out[['n_events','duration']].describe())

# 可选：查看前几行
print(out.head(10))


In [ ]:
# === 基于比例阈值的快照标注（简单版） ===
# 你用 KDE 得到合适的阈值后，把 THRESH_P 换成你的数即可
THRESH_P = 0.001  # 示例：恶意占比阈值（你会根据 KDE 来改，比如 0.05 / 0.1）

# 可选：防止“1~2个恶意点”误判（如果不需要，把 MIN_MAL 设为 1）
MIN_MAL = 3

# 为避免右边界遗漏，最后一个窗口用 [a, b] 其余用 [a, b)
out = out.copy()
n_rows = len(out)

n_mal_list = []
mal_ratio_list = []
snap_label_list = []

for i, row in out.iterrows():
    a, b = row['t_start'], row['t_end']
    if i < n_rows - 1:
        mask = (df['time_norm'] >= a) & (df['time_norm'] < b)  # 半开
    else:
        mask = (df['time_norm'] >= a) & (df['time_norm'] <= b) # 最后一个闭合到 b

    sub = df.loc[mask]
    n_total = len(sub) if 'n_events' not in row or pd.isna(row['n_events']) else int(row['n_events'])
    # 为稳妥，这里直接按实际筛选到的数量计算
    n_total = len(sub)

    n_mal = int((sub['label'] == 1).sum())
    mal_ratio = (n_mal / n_total) if n_total > 0 else 0.0
    is_mal = int((mal_ratio >= THRESH_P) and (n_mal >= MIN_MAL))

    n_mal_list.append(n_mal)
    mal_ratio_list.append(mal_ratio)
    snap_label_list.append(is_mal)

out['n_mal'] = n_mal_list
out['mal_ratio'] = mal_ratio_list
out['snapshot_label'] = snap_label_list  # 1=恶意快照, 0=正常快照

print(out[['t_start','t_end','n_events','n_mal','mal_ratio','snapshot_label']].head(10))
print("恶意快照个数：", out['snapshot_label'].sum(), "/", len(out))

# 可选：保存
out.to_csv('snapshot_equal_count_by_region_normal.csv', index=False)


#### 时间特征编码

In [ ]:
import torch
import numpy as np

class TimeEncode(torch.nn.Module):
    def __init__(self, expand_dim, factor=5):
        super(TimeEncode, self).__init__()
        #init_len = np.array([1e8**(i/(time_dim-1)) for i in range(time_dim)])
        
        time_dim = expand_dim
        self.factor = factor
        self.basis_freq = torch.nn.Parameter((torch.from_numpy(1 / 10 ** np.linspace(0, 9, time_dim))).float())
        self.phase = torch.nn.Parameter(torch.zeros(time_dim).float())
        
        #self.dense = torch.nn.Linear(time_dim, expand_dim, bias=False)
        #torch.nn.init.xavier_normal_(self.dense.weight)
        
    def forward(self, ts):
        # ts: [N, L]
        batch_size = ts.size(0)
        seq_len = ts.size(1)
                
        ts = ts.view(batch_size, seq_len, 1)# [N, L, 1]
        map_ts = ts * self.basis_freq.view(1, 1, -1) # [N, L, time_dim]
        map_ts += self.phase.view(1, 1, -1)
        
        harmonic = torch.cos(map_ts)
        return harmonic #self.dense(harmonic)

# 创建时间编码器
time_encoder = TimeEncode(expand_dim=8)

# 提取所有time_norm值并转换为tensor
# 注意：需要reshape为[N, 1]格式，因为每行只有一个时间值
time_values = torch.tensor(df['time_norm'].values, dtype=torch.float32).unsqueeze(1)  # shape: [N, 1]
print(f"\n时间tensor形状: {time_values.shape}")

# 对所有时间值进行编码
with torch.no_grad():  # 如果只是推理不需要梯度
    encoded_times = time_encoder(time_values)  # shape: [N, 1, 8]

print(f"编码后形状: {encoded_times.shape}")

# 由于每行只有一个时间值，我们可以去掉中间维度
encoded_times_squeezed = encoded_times.squeeze(1)  # shape: [N, 8]
print(f"压缩后形状: {encoded_times_squeezed.shape}")

# 将编码结果转换为numpy数组，方便后续使用
encoded_array = encoded_times_squeezed.numpy()
print(f"编码数组形状: {encoded_array.shape}")

# 可以将编码结果添加到DataFrame中
for i in range(8):
    df[f'time_enc_{i}'] = encoded_array[:, i]

print(f"\n添加编码列后的DataFrame形状: {df.shape}")
print("前5行的时间编码结果:")
print(df[['time_norm'] + [f'time_enc_{i}' for i in range(8)]].head())

# 或者直接获取编码矩阵用于其他用途
print(f"\n前5行的8维时间编码向量:")
for i in range(5):
    print(f"第{i+1}行 (time_norm={df.iloc[i]['time_norm']}): {encoded_array[i]}")

# 如果你想要批量处理，也可以这样使用：
print(f"\n批量编码示例 - 编码向量的统计信息:")
print(f"编码值范围: [{encoded_array.min():.4f}, {encoded_array.max():.4f}]")
print(f"编码值均值: {encoded_array.mean():.4f}")
print(f"编码值标准差: {encoded_array.std():.4f}")